# 01. Data Loading

> Load the raw `ECoG_Handpose.mat` file and expose helpers to access the time, 60-channel ECoG, paradigm-label, and 5-finger data-glove channels.

This module is the single entry point used by every downstream notebook so we never re-implement file I/O. Constants for sampling rate (1200 Hz) and channel layout live here.

In [ ]:
#| default_exp data

In [ ]:
#| hide
from nbdev.showdoc import *

## Constants

The .mat file stores everything in a single `y` matrix of shape `(67, T)` (channel × time). We slice it into the documented groups.

In [ ]:
#| export
from dataclasses import dataclass
from pathlib import Path
import numpy as np
from scipy.io import loadmat

In [ ]:
#| export
FS = 1200                       # sampling frequency, Hz
TIME_ROW   = 0                  # CH1   — sample time
ECOG_ROWS  = slice(1, 61)       # CH2-61 — 60 ECoG channels
LABEL_ROW  = 61                 # CH62  — paradigm label
GLOVE_ROWS = slice(62, 67)      # CH63-67 — 5 finger glove channels

GESTURE_NAMES = {0: 'relax', 1: 'fist', 2: 'peace', 3: 'open'}
FINGER_NAMES = ['thumb', 'index', 'middle', 'ring', 'little']

## Recording container

A small dataclass keeps the four logical signal groups together so downstream notebooks don't have to remember channel offsets.

In [ ]:
#| export
@dataclass
class ECoGRecording:
    "Container for the rock-paper-scissors ECoG dataset."
    time:   np.ndarray   # (T,)
    ecog:   np.ndarray   # (60, T)
    labels: np.ndarray   # (T,) int — paradigm code per sample
    glove:  np.ndarray   # (5, T) — finger flexion per sample
    fs:     float = FS

    @property
    def n_samples(self):  return self.ecog.shape[1]

    @property
    def duration_s(self): return self.n_samples / self.fs

## Loader

`load_ecog` walks up from the current working directory to locate `data/ECoG_Handpose.mat`, so it works whether you run from the repo root or from `nbs/`.

In [ ]:
#| export
def _find_default_path():
    rel = Path('data/ECoG_Handpose.mat')
    for parent in [Path.cwd(), *Path.cwd().parents]:
        if (parent / rel).exists(): return parent / rel
    raise FileNotFoundError(f'ECoG_Handpose.mat not found above {Path.cwd()}')

In [ ]:
#| export
def load_ecog(path=None):
    "Load `ECoG_Handpose.mat` and return an `ECoGRecording`."
    path = Path(path) if path else _find_default_path()
    y = loadmat(str(path))['y']
    return ECoGRecording(
        time   = y[TIME_ROW],
        ecog   = y[ECOG_ROWS],
        labels = y[LABEL_ROW].astype(int),
        glove  = y[GLOVE_ROWS],
    )

## Smoke test

Verify the loader returns the documented shapes and label values. Future notebooks should be able to `from br41n_ecog_hand_pose.data import load_ecog` and get the same object back.

In [ ]:
rec = load_ecog()
print(f'duration:    {rec.duration_s:.1f} s  ({rec.n_samples:,} samples @ {rec.fs} Hz)')
print(f'ecog shape:  {rec.ecog.shape}')
print(f'labels:      unique = {sorted(np.unique(rec.labels).tolist())}  ({GESTURE_NAMES})')
print(f'glove shape: {rec.glove.shape}  ({FINGER_NAMES})')

duration:    422.5 s  (507,025 samples @ 1200 Hz)
ecog shape:  (60, 507025)
labels:      unique = [0, 1, 2, 3]  ({0: 'relax', 1: 'fist', 2: 'peace', 3: 'open'})
glove shape: (5, 507025)  (['thumb', 'index', 'middle', 'ring', 'little'])


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()